# AI Agent Sandbox with Search, Calculator, arXiv & Weather Tools (v3)
This notebook demonstrates a fully working AI Agent using LangChain, Google Gemini (`gemini-3.5-flash-lite`), Google Serper Search, Calculator, arXiv paper research, and Real-Time Weather tools.

In [ ]:
# Dependencies check & load environment
from dotenv import load_dotenv
import os

load_dotenv()
print("Environment variables loaded successfully!")

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper

# Initialize & test Google Serper Search wrapper
search = GoogleSerperAPIWrapper()
result = search.run("Top 10 news of India Today")
print("Search Result Preview:", result[:300] + "...")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    api_key=os.getenv("GOOGLE_API_KEY")
)

# Test LLM directly
response = llm.invoke("Hello! Introduce yourself briefly.")
raw_content = response.content
clean_reply = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in raw_content) if isinstance(raw_content, list) else str(raw_content)
print("Gemini Response:", clean_reply)

In [ ]:
from langchain_core.tools import tool
import math

# Define Calculator Tool
@tool
def calculator(expression: str) -> str:
    """Calculates the result of a mathematical expression.
    Input should be a valid mathematical expression string (e.g. '2 + 2', '15 * 8', '100 / 4', '2**10', 'sqrt(144)').
    """
    try:
        allowed_names = {
            'abs': abs, 'round': round, 'pow': pow, 'min': min, 'max': max,
            'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
            'log': math.log, 'pi': math.pi, 'e': math.e
        }
        result = eval(expression, {'__builtins__': None}, allowed_names)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"

# Test calculator tool
print("Calculator output:", calculator.invoke("15 * 8 + 42"))

In [ ]:
from langchain_core.tools import tool
import arxiv

# Define arXiv Research Search Tool with Executive Professional Formatting
@tool
def arxiv_search(query: str) -> str:
    """Searches scientific research papers on arXiv.
    Input should be a search query topic or research domain (e.g. 'quantum computing', 'transformer models', 'artificial intelligence').
    Returns paper titles, authors, published dates, category, formatted abstract summary, and direct PDF download links.
    """
    try:
        client = arxiv.Client()
        search = arxiv.Search(query=query, max_results=3, sort_by=arxiv.SortCriterion.Relevance)
        results = list(client.results(search))
        if not results:
            return 'No research papers found for the query.'
        output = []
        for i, paper in enumerate(results, 1):
            author_names = [a.name for a in paper.authors[:3]]
            authors_str = ', '.join(author_names)
            if len(paper.authors) > 3:
                authors_str += ' et al.'
            clean_summary = ' '.join(paper.summary.replace('\n', ' ').split())
            if len(clean_summary) > 350:
                clean_summary = clean_summary[:350] + '...'
            pub_date = paper.published.strftime('%B %d, %Y')
            category = getattr(paper, 'primary_category', 'Research')
            paper_id = paper.entry_id.split('/')[-1]

            formatted_paper = (
                f"### 📄 {i}. {paper.title.strip()}\n"
                f"- **👥 Authors:** {authors_str}\n"
                f"- **📅 Published:** {pub_date} | **🏷️ Category:** `{category}`\n"
                f"- **🔗 PDF Document:** [{paper_id}.pdf]({paper.pdf_url})\n\n"
                f"**📝 Abstract Summary:**\n"
                f"> {clean_summary}"
            )
            output.append(formatted_paper)
        return '\n\n---\n\n'.join(output)
    except Exception as e:
        return f"Error searching arXiv: {e}"

# Test arXiv search tool standalone
print("arXiv Search Test:\n", arxiv_search.invoke("quantum computing"))

In [ ]:
from langchain_core.tools import tool
import requests

# Define Real-Time Weather Tool using Open-Meteo API (No API key required)
@tool
def get_weather(city: str) -> str:
    """Fetches real-time weather information for a given city name.
    Input should be a city name (e.g. 'Mumbai', 'London', 'New York', 'Tokyo').
    Returns current temperature in °C, humidity percentage, and wind speed.
    """
    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1&language=en&format=json"
        geo_res = requests.get(geo_url, timeout=5).json()
        if not geo_res.get("results"):
            return f"Could not find location coordinates for '{city}'."
        
        loc = geo_res["results"][0]
        lat, lon = loc["latitude"], loc["longitude"]
        city_name = loc.get("name", city)
        country = loc.get("country", "")

        weather_url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m,relative_humidity_2m,wind_speed_10m"
        w_res = requests.get(weather_url, timeout=5).json()
        current = w_res.get("current", {})

        temp = current.get("temperature_2m")
        humidity = current.get("relative_humidity_2m")
        wind = current.get("wind_speed_10m")

        return f"Weather in {city_name}, {country}: Temperature: {temp}°C, Humidity: {humidity}%, Wind Speed: {wind} km/h."
    except Exception as e:
        return f"Error fetching weather data: {e}"

# Test weather tool standalone
print("Weather Tool Output:", get_weather.invoke("Mumbai"))

In [ ]:
from langchain_core.tools import tool
import trafilatura

# Define URL Web Page Reader & Summarizer Tool using Trafilatura
@tool
def summarize_url(url: str) -> str:
    """Fetches, extracts, and summarizes the main text content of any website or URL.
    Input should be a valid web page URL string (e.g. 'https://en.wikipedia.org/wiki/Artificial_intelligence' or 'https://news.ycombinator.com').
    Returns extracted readable text content from the web page.
    """
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return f"Error: Could not fetch content from URL '{url}'. Please check if the URL is valid and accessible."
        
        extracted_text = trafilatura.extract(downloaded, include_comments=False, include_tables=True, no_fallback=False)
        if not extracted_text:
            return f"Error: No readable text content could be extracted from '{url}'."
        
        clean_text = ' '.join(extracted_text.split())
        truncated_text = clean_text[:3500]
        if len(clean_text) > 3500:
            truncated_text += " ... [Content truncated for length]"
            
        return (
            f"### 🌐 Extracted Content from URL:\n"
            f"**URL:** [{url}]({url})\n\n"
            f"```text\n"
            f"{truncated_text}\n"
            f"```"
        )
    except Exception as e:
        return f"Error extracting content from URL: {e}"

# Test URL summarizer tool standalone
print("URL Summarizer Output:\n", summarize_url.invoke("https://en.wikipedia.org/wiki/Artificial_intelligence")[:400])

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# Create Agent with Search, Calculator, arXiv, Weather, and URL Summarizer tools
agent = create_agent(
    model=llm,
    tools=[search.run, calculator, arxiv_search, get_weather, summarize_url],
    system_prompt=(
        "You are a helpful AI research assistant equipped with Google Search, a Calculator, an arXiv Research tool, a Weather tool, and a URL Summarizer tool. "
        "Use Google Search for real-time web news, Calculator for math calculations, arXiv Search for scientific papers, Weather for city weather, and summarize_url to read web pages."
    ),
    checkpointer=MemorySaver()
)
print("Agent successfully created with 5 tools (Search, Calculator, arXiv, Weather, URL Summarizer)!")

In [ ]:
# Test invoking the Agent with weather question
question = "What is the current weather in Tokyo and Mumbai?"
config = {"configurable": {"thread_id": "Prathamesh_Session_1"}}

response = agent.invoke({"messages": [{"role": "user", "content": question}]}, config)
raw_reply = response["messages"][-1].content
clean_reply = "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in raw_reply) if isinstance(raw_reply, list) else str(raw_reply)
print("Agent Response:\n", clean_reply)